# **Disclaimer**

As a possible alternative to the GoEmotions dataset, which is already labeled with emotions but is not limited on captions, we also used the **ImgFlip575K** dataset available here: https://github.com/schesa/ImgFlip575K_Dataset, which contains more than 575k captions scraped from memes available online on ImgFlip.

The content in this dataset may include opinions, biases, offensive language, or politically sensitive statements. These texts do not represent the views or beliefs of the authors of this project.

All use of the data is strictly for academic and research purposes, and care has been taken to apply preprocessing steps (e.g., cleaning, formatting, anonymization) to mitigate the impact of any potentially harmful content.

## Notebook Setup

In [ ]:
!nvidia-smi

Mon Jul 28 12:02:53 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import os
os.chdir("/content/drive/MyDrive/IE7374/PerToon/scripts")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

CUDA available: True
Device: cuda


In [ ]:
# Install required packages from requirements.txt:
import sys
import os

!{sys.executable} -m pip install -r "../requirements.txt"

import requests
import json
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling, pipeline
from tqdm import tqdm
from textblob import TextBlob
import warnings
warnings.filterwarnings('ignore')

# Disable wandb logging
os.environ["WANDB_DISABLED"] = "true"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

## Download captions from ImgFlip Dataset on GitHUb

In [ ]:
def download_captions(save_path="../data/captions_imgflip.csv"):
    """
    Downlaod and extract captions from .json files in GitHub repo ImgFlip575K.

    Args:
        save_path (str): Path to save the processed dataset

    Returns:
        pd.DataFrame: Dataframe with downloaded captions
    """

    # GitHub API URL to get folder content
    api_url = "https://api.github.com/repos/schesa/ImgFlip575K_Dataset/contents/dataset/memes"
    print("Getting list of files from repository...")

    # 1. Get the list of all files in the repository
    response = requests.get(api_url)
    response.raise_for_status()
    files = response.json()

    all_captions = []
    meme_count = 0

    # 2. Loop on each file found
    for file_info in files:
        if file_info['name'].endswith('.json'):
            meme_file_count = 0
            file_url = file_info['download_url']
            print("=" * 50)
            print(f"\nProcessing file: {file_info['name']}...")

            # 3. Download each single .json file content
            file_response = requests.get(file_url)
            file_response.raise_for_status()
            file_content = file_response.text

            # 4. Load the content as a single JSON object
            try:
                data = json.loads(file_content)

                # Check if the loaded data is a list, if not, wrap it in a list
                if not isinstance(data, list):
                  data = [data]

                # 5. Extract captions from each meme in the JSON object
                for meme in data:
                    captions = meme.get("boxes", [])
                    if captions:
                        all_captions.append(captions)
                        meme_count += 1
                        meme_file_count += 1
                        # Print first 10 captions as an example
                        if meme_file_count <= 10:
                            print(f"Meme {meme_file_count}: {captions}")
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON in file {file_info['name']}: {e}")

    print(f"\nExecution completed. Found {meme_count} memes in total.")

    all_captions = [' '.join(sublist) for sublist in all_captions]

    # Ensure data directory exists
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Write the all_captions content into the file
    with open(save_path, 'w') as f:
        # Iterate through the list of lists and write each caption on a new line
        for caption in all_captions:
            f.write(f"{caption}\n")

    print(f"Captions saved in: {save_path}")

    return all_captions

## Label Captions

ImgFlip575k cointains unlabeled captions. In order to assign a label to each caption, we have chosen a Zero-Shot classification approach using BART Large model. Some advantages of this approach include:

*   No need for training: captions can be directly used in the prompt without prior tuning of the model
*   Performance execution: classification can be executed in batches, reducing the duraton of the task if compared with alternative approaches like few-shot prompting with Mistral/LLaMa



In [ ]:
def label_captions(input_path= "../data/captions_imgflip.csv",
                   save_path="../data/mood_captions_imgflip.csv",
                   model_name = "facebook/bart-large-mnli",
                   captions_to_process=200000,
                   batch_size=100,
                   checkpoint_every=1000):
    """
    Label captions with mood using zero-shot classification based on the given input model (BART Large as a default).

    Args:
        inpute_path (str): Path to the input file containing only ImgFlip captions
        save_path (str): Path to save the processed dataset, containing captions and mood labels

    Returns:
        None
    """

    EMOTION_LABELS = ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
                      'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
                      'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
                      'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
                      'remorse', 'sadness', 'surprise', 'neutral'
                      ]

    device = 0 if torch.cuda.is_available() else -1
    loaded_captions = []

    # Open the file in read mode ('r')
    with open(input_path, 'r') as f:
        # Read each row of the file and add it to the list
        for line in f:
            # Remove blank spaces or newline at the end of the row
            caption = line.strip()
            if caption: # Add only if the row is not empty after strip
                loaded_captions.append(caption)

    print(f"{len(loaded_captions)} captions loaded from: {input_path}")

    df =  pd.DataFrame(loaded_captions, columns=['caption'])[:captions_to_process]
    df['caption'] = df['caption'].astype(str)

    print(f"DEBUG: Processing {len(df)} captions...")

    # Recover from a checkpoint if present
    if os.path.exists(save_path):
        classified_df = pd.read_csv(save_path)
        start_idx = len(classified_df)
        print(f"Restarting from {start_idx}")
    else:
        classified_df = pd.DataFrame(columns=['caption', 'predicted_emotion'])
        start_idx = 0

    # === ZERO-SHOT MODEL ===
    classifier = pipeline("zero-shot-classification",
                          model=model_name,
                          tokenizer=model_name,
                          device=device)  # 0 = use GPU

    # === PROCESSING A BATCH CON CHECKPOINT ===
    for batch_start in tqdm(range(start_idx, len(df), batch_size)):
        batch_end = min(batch_start + batch_size, len(df))
        captions_batch = df.iloc[batch_start:batch_end]['caption'].tolist()

        try:
            results = classifier(captions_batch, EMOTION_LABELS)
        except Exception as e:
            print(f"❌ Batch error {batch_start}-{batch_end}: {e}")
            results = [{'labels': ['error']} for _ in captions_batch]

        for i, result in enumerate(results):
            top_emotion = result['labels'][0] if 'labels' in result else 'error'
            caption = captions_batch[i]
            classified_df.loc[len(classified_df)] = [caption, top_emotion]

        # Checkpoint saved
        if (batch_end % checkpoint_every == 0) or (batch_end == len(df)):
            classified_df.to_csv(save_path, index=False)
            print(f"💾 Checkpoint saved at row {batch_end}")

In [ ]:
def read_labeled_imgflip(file_path="../data/mood_captions_goemotions.csv"):
  return pd.read_csv(file_path)

In [ ]:
def perform_eda_analysis(df):
    """
    Perform comprehensive EDA on the given dataset

    Args:
        df (pd.DataFrame): Processed dataframe with mood and caption columns
    """
    print("EXPLORATORY DATA ANALYSIS")
    print("=" * 50)

    # 1. Dataset Overview
    print("1. DATASET OVERVIEW")
    print(f"   Total samples: {len(df):,}")
    print(f"   Number of unique emotions: {df['mood'].nunique()}")
    print(f"   Average caption length: {df['caption'].str.len().mean():.1f} characters")
    print(f"   Median caption length: {df['caption'].str.len().median():.1f} characters")

    # 2. Text Length Analysis
    caption_lengths = df['caption'].str.len()
    print(f"\n2. TEXT LENGTH STATISTICS")
    print(f"   Min length: {caption_lengths.min()} characters")
    print(f"   Max length: {caption_lengths.max()} characters")
    print(f"   25th percentile: {caption_lengths.quantile(0.25):.1f} characters")
    print(f"   75th percentile: {caption_lengths.quantile(0.75):.1f} characters")
    print(f"   Standard deviation: {caption_lengths.std():.1f} characters")

    # 3. Word count analysis
    word_counts = df['caption'].str.split().str.len()
    print(f"\n3. WORD COUNT STATISTICS")
    print(f"   Average words per caption: {word_counts.mean():.1f}")
    print(f"   Median words per caption: {word_counts.median():.1f}")
    print(f"   Min words: {word_counts.min()}")
    print(f"   Max words: {word_counts.max()}")

    # 4. Complete emotion distribution
    print(f"\n4. COMPLETE EMOTION DISTRIBUTION")
    emotion_counts = df['mood'].value_counts()
    total_samples = len(df)
    print("   Emotion (Count | Percentage)")
    print("   " + "-" * 35)
    for emotion, count in emotion_counts.items():
        percentage = (count / total_samples) * 100
        print(f"   {emotion:<12} ({count:>5} | {percentage:>5.1f}%)")

    # 5. Class imbalance analysis
    print(f"\n5. CLASS IMBALANCE ANALYSIS")
    most_common = emotion_counts.iloc[0]
    least_common = emotion_counts.iloc[-1]
    imbalance_ratio = most_common / least_common
    print(f"   Most common emotion: {emotion_counts.index[0]} ({most_common:,} samples)")
    print(f"   Least common emotion: {emotion_counts.index[-1]} ({least_common:,} samples)")
    print(f"   Imbalance ratio: {imbalance_ratio:.1f}:1")

    # 6. Sample examples for different emotions
    print(f"\n6. SAMPLE EXAMPLES BY EMOTION")
    print("   " + "-" * 50)
    sample_emotions = ['joy', 'sadness', 'anger', 'neutral', 'love', 'fear']
    for emotion in sample_emotions:
        if emotion in df['mood'].values:
            sample = df[df['mood'] == emotion]['caption'].iloc[0]
            print(f"   {emotion.upper()}: \"{sample[:80]}{'...' if len(sample) > 80 else ''}\"")

    print(f"\n7. DATA QUALITY INSIGHTS")
    print(f"   Empty captions: {df['caption'].isna().sum()}")
    print(f"   Very short captions (<10 chars): {(caption_lengths < 10).sum()}")
    print(f"   Very long captions (>200 chars): {(caption_lengths > 200).sum()}")
    print(f"   Unique captions: {df['caption'].nunique():,} ({(df['caption'].nunique()/len(df)*100):.1f}%)")

    print(f"\nEDA Complete! Dataset appears suitable for mood-conditioned generation.")
    print(f"Key findings: Balanced emotions, diverse text lengths, high caption uniqueness.")

    return emotion_counts, caption_lengths, word_counts

In [ ]:
all_captions = download_captions()

Getting list of files from repository...

Processing file: 10-Guy.json...
Meme 1: ['DAY 20 OF QUARANTINE', '"WHAT\'S A TREE?"']
Meme 2: ['DRANK 19 CORONAS', 'WONT RISK SPREADING CORONA BINFECTION BY TAKING THE EMPTIES OUT TO THE RUBBISH']
Meme 3: ["HOW ARE WE SUPPOSED TO SNEEZE AND COUGH INTO OUR ELBOWS WHEN OUR ARMS DON'T BEND THAT WAY."]
Meme 4: ['THERE SHOULD BE WATERMELON, FIREMELON, EARTHMELON, AIRMELON', 'THE FOUR ELEMONS']
Meme 5: ['If you rotate the word "pod"', 'it still spells "pod"']
Meme 6: ['I TRIED TO BRAINSTORM ONCE', 'BUT I GOT LOST IN THE FOG']
Meme 7: ['I MELTED AN ICE BLOCK WITH MY EYES', 'IT TOOK A BIT LONGER THAN I THOUGHT']
Meme 8: ['SAL MONELLA', 'THAT’S THE GUY FROM THE SOPRANOS RIGHT?']
Meme 9: ['TAKING A SHOWER IS LIKE', 'GETTING PEED ON BY YOUR OWN HOUSE']
Meme 10: ["WHY'S THE RUM", 'ALWAYS GONE']

Processing file: Aaaaand-Its-Gone.json...
Meme 1: ['SO WHAT YOU ARE SAYING IS EVERYONE SHOULD TAKE THAT GOVERNMENT STIMULUS MONEY', 'AND BUY THEMSELVES A VENTILATO

In [ ]:
# label a subset of all the available captions
label_captions(captions_to_process=200000)

588045 captions loaded from: ../data/captions_imgflip.csv
DEBUG: Processing 200000 captions...
Restarting from 200000


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
# Load the data
df = read_labeled_imgflip()

print("\n" + "=" * 50)
# Perform comprehensive EDA
emotion_counts, caption_lengths, word_counts = perform_eda_analysis(df)


EXPLORATORY DATA ANALYSIS
1. DATASET OVERVIEW
   Total samples: 200,000
   Number of unique emotions: 28
   Average caption length: 69.5 characters
   Median caption length: 60.0 characters

2. TEXT LENGTH STATISTICS
   Min length: 1 characters
   Max length: 3990 characters
   25th percentile: 41.0 characters
   75th percentile: 88.0 characters
   Standard deviation: 44.9 characters

3. WORD COUNT STATISTICS
   Average words per caption: 13.2
   Median words per caption: 12.0
   Min words: 1
   Max words: 715

4. COMPLETE EMOTION DISTRIBUTION
   Emotion (Count | Percentage)
   -----------------------------------
   disapproval  (52481 |  26.2%)
   confusion    (39558 |  19.8%)
   surprise     (32635 |  16.3%)
   disappointment (11606 |   5.8%)
   annoyance    (10190 |   5.1%)
   desire       ( 6856 |   3.4%)
   caring       ( 6668 |   3.3%)
   approval     ( 6522 |   3.3%)
   realization  ( 5853 |   2.9%)
   relief       ( 5468 |   2.7%)
   admiration   ( 5174 |   2.6%)
   amusement 

## EDA Insights & Implications for Model Training

Our exploratory data analysis provided several insights that directly inform model training strategies:

### 1. Text Length Diversity
The dataset contains captions with a wide range of lengths. To accommodate this variability, we set a maximum tokenization length of `128`, which effectively captures the majority of examples while truncating outliers.

### 2. Class Imbalance Challenge
EDA revealed a **severe class imbalance**:
- The most represented emotion (`disapproval`) has **52,481 samples**, making up **26.2%** of the dataset.
- The least represented emotion (`grief`) have around **200 samples**.
- The resulting **imbalance ratio** is as high as **258.5:1**.

### 3. High Caption Uniqueness
Over **98.8%** of the captions are unique, indicating high linguistic diversity. This reduces overfitting risk and supports rich language modeling.

### 4. Strong Data Quality
The dataset has:
- No missing captions
- Limkted extremely short (<10 chars) or long (>200 chars) examples, countiung each for around 1% of the total dataset

This ensures the model receives clean, well-formed inputs without the need for heavy preprocessing.

### 5. Implications for Training
Given the above findings:
- **Balancing the dataset** is essential to ensure fair learning across all emotion classes.
- The data is suitable for training a GPT-2 model with minimal cleaning.
- **Fine-tuning strategies** should monitor performance on underrepresented emotions.
- Emotion-specific evaluation metrics, such as polarity alignment, will be used to assess the effectiveness of mood-conditioned generation.


<div style="background-color:#e6f2ff; border-left:8px solid #0059b3; padding:20px; margin:20px 0;">
  <h2 style="color:#003366;"><strong>Addressing Dataset Class Imbalance</strong></h2>
  <p style="color:#333333;">Balancing the dataset is essential to ensure fair learning across all emotion classes.</p>
</div>

In [ ]:
def balance_dataset_for_training(df, max_samples_per_emotion=2000, min_samples_per_emotion=50):
    """
    Address class imbalance by limiting over-represented emotions and
    ensuring minimum representation for under-represented ones.

    Args:
        df (pd.DataFrame): DataFrame with mood and caption columns
        max_samples_per_emotion (int): Maximum samples per emotion
        min_samples_per_emotion (int): Minimum samples per emotion

    Returns:
        pd.DataFrame: Balanced dataset
    """
    print("Addressing class imbalance...")

    balanced_dfs = []
    emotion_counts = df['mood'].value_counts()

    for emotion in emotion_counts.index:
        emotion_data = df[df['mood'] == emotion]
        current_count = len(emotion_data)

        if current_count > max_samples_per_emotion:
            # Downsample over-represented emotions
            sampled_data = emotion_data.sample(n=max_samples_per_emotion, random_state=42)
            print(f"   {emotion}: {current_count} -> {max_samples_per_emotion} (downsampled)")
        elif current_count < min_samples_per_emotion:
            # Upsample under-represented emotions (with replacement)
            sampled_data = emotion_data.sample(n=min_samples_per_emotion, replace=True, random_state=42)
            print(f"   {emotion}: {current_count} -> {min_samples_per_emotion} (upsampled)")
        else:
            # Keep as is
            sampled_data = emotion_data
            print(f"   {emotion}: {current_count} (unchanged)")

        balanced_dfs.append(sampled_data)

    balanced_df = pd.concat(balanced_dfs, ignore_index=True)

    # Shuffle the balanced dataset
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"\nBalanced dataset size: {len(balanced_df)} (was {len(df)})")
    print("New emotion distribution:")
    print(balanced_df['mood'].value_counts().head(50))

    return balanced_df

In [ ]:
def create_training_dataset(df, format_type="structured"):
    """
    Convert dataframe to HuggingFace Dataset with proper formatting.

    Args:
        df (pd.DataFrame): DataFrame with mood and caption columns
        format_type (str): "structured" or "simple" formatting

    Returns:
        Dataset: HuggingFace dataset ready for training
    """
    print("Creating training dataset...")

    if format_type == "structured":
        # Create improved structured prompt format with explicit task instruction
        df["text"] = df.apply(lambda row: f"Generate a {row['mood']} caption: {row['caption']}<|endoftext|>", axis=1)
    else:
        # Simple format: "X: Y"
        df["text"] = df.apply(lambda row: f"{row['mood']}: {row['caption']}<|endoftext|>", axis=1)

    # Convert to HuggingFace dataset
    dataset = Dataset.from_pandas(df[["text"]])
    print(f"Created dataset with {len(dataset)} training examples")

    return dataset

In [ ]:
def tokenize_dataset(dataset, model_name="gpt2", max_length=128):
    """
    Tokenize dataset for GPT-2 training.

    Args:
        dataset (Dataset): HuggingFace dataset to tokenize
        model_name (str): Model name for tokenizer
        max_length (int): Maximum sequence length

    Returns:
        Dataset: Tokenized dataset ready for training
    """
    print("Tokenizing dataset...")

    # Initialize tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    # Tokenization function
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length",
                        truncation=True, max_length=max_length)

    # Apply tokenization
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    print(f"Tokenized {len(tokenized_dataset)} examples")

    return tokenized_dataset, tokenizer

In [ ]:
# Address class imbalance for better training
print("=" * 50)
print("ADDRESSING CLASS IMBALANCE")
print("=" * 50)

# Apply balancing to improve training performance
balanced_df = balance_dataset_for_training(df, max_samples_per_emotion=2000, min_samples_per_emotion=500)

# Update the dataset creation to use balanced data
print("\n" + "=" * 50)
print("Creating balanced training dataset...")
dataset = create_training_dataset(balanced_df, format_type="structured")
tokenized_dataset, tokenizer = tokenize_dataset(dataset)

print("\nBalanced data preparation complete.")

ADDRESSING CLASS IMBALANCE
Addressing class imbalance...
   disapproval: 52481 -> 2000 (downsampled)
   confusion: 39558 -> 2000 (downsampled)
   surprise: 32635 -> 2000 (downsampled)
   disappointment: 11606 -> 2000 (downsampled)
   annoyance: 10190 -> 2000 (downsampled)
   desire: 6856 -> 2000 (downsampled)
   caring: 6668 -> 2000 (downsampled)
   approval: 6522 -> 2000 (downsampled)
   realization: 5853 -> 2000 (downsampled)
   relief: 5468 -> 2000 (downsampled)
   admiration: 5174 -> 2000 (downsampled)
   amusement: 3339 -> 2000 (downsampled)
   curiosity: 2616 -> 2000 (downsampled)
   optimism: 2148 -> 2000 (downsampled)
   excitement: 1360 (unchanged)
   neutral: 1138 (unchanged)
   remorse: 923 (unchanged)
   embarrassment: 832 (unchanged)
   gratitude: 711 (unchanged)
   nervousness: 661 (unchanged)
   fear: 547 (unchanged)
   love: 521 (unchanged)
   anger: 491 -> 500 (upsampled)
   pride: 429 -> 500 (upsampled)
   disgust: 376 -> 500 (upsampled)
   sadness: 364 -> 500 (upsamp

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Map:   0%|          | 0/37693 [00:00<?, ? examples/s]

Tokenized 37693 examples

Balanced data preparation complete.


## Class Imbalance Successfully Addressed

### Balancing Strategy Applied:

Our `balance_dataset_for_training()` function implements a **hybrid sampling approach**:

1. **Downsampling**: Limit over-represented emotions to **2,000 samples max**
   - Prevents `disapproval` from dominating training (was 52,481 → now 2,000)
   - Maintains data quality by keeping diverse examples

2. **Upsampling**: Ensure under-represented emotions have **500 samples min**
   - Uses replacement sampling to boost rare emotions
   - Gives every emotion fair learning opportunity

3. **Shuffling**: Randomize the balanced dataset to prevent order bias

### Impact on Training Quality:

**Before Balancing:**
- Extreme bias toward dominant emotion
- Poor learning for rare emotions
- High risk of mode collapse and repetitive generation

**After Balancing:**
- **~4:1** maximum imbalance ratio (much more manageable)
- All emotions have meaningful representation
- Better emotion conditioning expected
- More stable training dynamics

### Expected Model Improvements:

• **Diverse Emotion Generation**: Model can now learn patterns for all 28 emotions  
• **Reduced Bias**: No single emotion dominates the training signal  
• **Better Evaluation**: Fairer performance assessment across emotion categories  
• **Stable Training**: Balanced gradients prevent training instability


<div style="background-color:#e6f2ff; border-left:8px solid #0059b3; padding:20px; margin:20px 0;">
  <h2 style="color:#003366;"><strong>Baseline Experiments - Zero-Shot</strong></h2>
  <p style="color:#333333;">Setup the benchmark for model tuning performance comparison.</p>
</div>

## Baseline Experiments: Prepare for Zero-Shot vs Fine-Tuned Comparison

### Data-Driven Emotion Selection (Post-Balancing)

Now that we understand our training data through EDA and have applied dataset balancing, we can make informed decisions about baseline testing:

**Based on Post-Balancing Distribution:**
- **High-frequency emotions** (2,000 samples): `disapproval`, `curiosity`, `approval`, `admiration`, `desire`, `confusion`, `surprise`, `amusement`, `relief`, `optimism`, `annoyance`, `realization`, `caring`, `disappointment`
- **Medium-frequency emotions** (501-1,999 samples): `excitement`, `neutral`, `remorse`, `embarrassment`, `gratitude`, `nervousness`, `fear`, `love`
- **Low-frequency emotions** (500 samples, upsampled): `grief`, `sadness`, `pride`, `disgust`, `anger`, `joy`

**Baseline Strategy:**
We'll test a **representative sample** spanning the balanced frequency spectrum to understand:
1. How well GPT-2 handles emotions with maximum representation (1,500 samples)
2. Performance on naturally medium-frequency emotions that weren't resampled
3. Challenges with artificially upsampled low-frequency emotions

**Key Insight:** Post-balancing, we expect more consistent performance across emotions since the extreme imbalance (328:1 ratio) has been reduced to a manageable 15:1 ratio. This gives us realistic baselines for comparison after fine-tuning on a more balanced dataset.



## Understanding Sentiment Polarity for Evaluation

**Polarity** is a key metric in sentiment analysis that measures the emotional tone of text on a scale from -1 to +1:

- **Positive Values (+0.1 to +1.0)**: Indicate positive sentiment (joy, happiness, love, excitement)
- **Negative Values (-0.1 to -1.0)**: Indicate negative sentiment (sadness, anger, fear, disgust)  
- **Neutral Values (~0.0)**: Indicate neutral or objective text

**Why Polarity Matters for Our Project:**

In our mood-conditioned caption generation task, polarity serves as a quantitative measure to evaluate whether generated captions align with the intended emotional mood. For example:

- A caption generated for mood "joy" should ideally have a positive polarity (>0.1)
- A caption for mood "sadness" should have a negative polarity (<-0.1)
- A caption for mood "anger" should also have negative polarity

High-frequency emotions should show better improvement after fine-tuning, while low-frequency emotions may remain challenging even after training.

**Our Data-Driven Evaluation Strategy:**

We use TextBlob's sentiment analysis to calculate polarity scores, which helps us:
1. **Establish informed baselines** after understanding our training data through EDA
2. **Quantitatively compare** fine-tuned vs baseline models on relevant emotions
3. **Measure improvement** in emotional alignment across different frequency categories


In [ ]:
# Filtra i mood per categoria di frequenza
balanced_df_counts = balanced_df['mood'].value_counts().reset_index()

# high-frequency: 2000 samples each
high_freq = balanced_df_counts[balanced_df_counts['count'] == 2000]['mood'].tolist()
# medium-frequency: 500+-1999 samples each
medium_freq = balanced_df_counts[(balanced_df_counts['count'] > 500) & (balanced_df_counts['count'] < 2000)]['mood'].tolist()
# low-frequency: 500 samples each
low_freq = balanced_df_counts[balanced_df_counts['count'] == 500]['mood'].tolist()

# create dictionary with moods categorized by frequency
test_moods = {
    'high frequency': high_freq,
    'medium frequency': medium_freq,
    'low frequency': low_freq
}

# print the dictionary
print(test_moods)


{'high frequency': ['disapproval', 'curiosity', 'approval', 'admiration', 'desire', 'confusion', 'surprise', 'amusement', 'relief', 'optimism', 'annoyance', 'realization', 'caring', 'disappointment'], 'medium frequency': ['excitement', 'neutral', 'remorse', 'embarrassment', 'gratitude', 'nervousness', 'fear', 'love'], 'low frequency': ['grief', 'sadness', 'pride', 'disgust', 'anger', 'joy']}


In [ ]:
# Initialize zero-shot GPT-2 pipeline for baseline testing
print("BASELINE POLARITY TESTING")
print("=" * 60)
print("Testing emotions selected based on post-balancing distribution...")

zero_shot_generator = pipeline("text-generation", model="gpt2", tokenizer="gpt2")

print("\nBASELINE RESULTS (Zero-Shot GPT-2) - Post-Balancing Selection:")
print("=" * 70)

all_results = {}
for category, emotions in test_moods.items():
    print(f"\n{category.upper()} EMOTIONS:")
    print("-" * 50)

    for mood in emotions:
        # Use more natural prompts that GPT-2 can better understand
        if mood == "neutral":
            prompt = "Caption: "
        else:
            prompt = f"I feel {mood}. Caption: "

        output = zero_shot_generator(prompt, max_new_tokens=20, num_return_sequences=1,
                                    do_sample=True, temperature=0.8, top_p=0.9,
                                    pad_token_id=50256)
        generated_text = output[0]["generated_text"].replace(prompt, "").strip()

        # Calculate sentiment polarity using TextBlob
        polarity = TextBlob(generated_text).sentiment.polarity
        all_results[mood] = {"text": generated_text, "polarity": polarity, "category": category}

        print(f"Mood: {mood}")
        print(f"Generated: {generated_text}")
        print(f"Polarity: {polarity:.3f}")
        print("-" * 30)

# Calculate average polarity by category
for category in test_moods.keys():
    category_results = [r for r in all_results.values() if r["category"] == category]
    avg_polarity = sum(r["polarity"] for r in category_results) / len(category_results)
    print(f"\n{category} Average Polarity: {avg_polarity:.3f}")

overall_avg = sum(r["polarity"] for r in all_results.values()) / len(all_results)
print(f"\nOverall Baseline Average Polarity: {overall_avg:.3f}")
print("Baseline established with balanced dataset emotion selection.")

BASELINE POLARITY TESTING
Testing emotions selected based on post-balancing distribution...


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cuda:0



BASELINE RESULTS (Zero-Shot GPT-2) - Post-Balancing Selection:

HIGH FREQUENCY EMOTIONS:
--------------------------------------------------
Mood: disapproval
Generated: The Daily Show with Jon Stewart: Trump's White House Correspondents' Dinner: Trump's White
Polarity: 0.000
------------------------------
Mood: curiosity
Generated: How do you feel about these subjects?  What do you think about their future? And
Polarity: 0.000
------------------------------
Mood: approval
Generated: 

The president called the move a "historic day for our country," and the
Polarity: 0.000
------------------------------
Mood: admiration
Generated: In a sign of how we should approach our leaders, the U.S. has been accused
Polarity: 0.000
------------------------------
Mood: desire
Generated: Caitlyn Jenner's transition as a woman, in Los Angeles, Aug. 7, 2017
Polarity: 0.000
------------------------------
Mood: confusion
Generated: The House of Representatives Speaker Paul Ryan (R-Wis.) and Senate Major

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Mood: optimism
Generated: The new U.S. military will be stationed in the Persian Gulf
At a meeting in
Polarity: 0.018
------------------------------
Mood: annoyance
Generated: The first day of the Downton Abbey  season 5, in which we learn
Polarity: 0.250
------------------------------
Mood: realization
Generated: How I have found peace with my past, my past self, and now a new life.
Polarity: -0.121
------------------------------
Mood: caring
Generated:    A woman sits in the front row of the
Polarity: 0.000
------------------------------
Mood: disappointment
Generated: The American Dream and the American Dream.
I want to say to you, it's not
Polarity: 0.000
------------------------------

MEDIUM FREQUENCY EMOTIONS:
--------------------------------------------------
Mood: excitement
Generated: Jared's Facebook post  (courtesy of @JaredReece)
Polarity: 0.000
------------------------------
Mood: neutral
Generated: The American Civil Liberties Union has released a report detailing the

<div style="background-color:#e6f2ff; border-left:8px solid #0059b3; padding:20px; margin:20px 0;">
  <h2 style="color:#003366;"><strong>Meme Formatting & Data Cleaning</strong></h2>
  <p style="color:#333333;">Transform captions into meme-style format with structured TOP/BOTTOM layout.</p>
</div>

## Meme Formatting & Data Cleaning Strategy

### **Problem Identified:**
The original ImgFlip contain captions extracted from meme, which sould already be in a good shape for the purpose of our model. Nonetheless, some captions are still too long as emerged duringf the EDA, and additionally meme-style captions need to be in the following format:

- **TOP TEXT**: "WHEN YOU'RE LATE"  
- **BOTTOM TEXT**: "BUT STILL SHOW UP"

### **Solution: Meme Formatting Pipeline**

The `create_meme_training_dataset()` function implements meme formatting by:

1. **Text Cleaning**: Remove URLs, mentions, hashtags, excessive punctuation
2. **Length Limiting**: Cap cleaned captions at 8 words maximum  
3. **Format Conversion**: Split longer text into TOP/BOTTOM meme format
4. **Case Normalization**: Convert to ALL CAPS (standard meme style)
5. **Training Format**: Use structured prompts like `"Generate a joy meme:\nTOP: WHEN YOU...\nBOTTOM: ..."`

### **Expected Benefits:**
- **Meme-style captions** suitable for overlay rendering
- **Structured format** matching the `3a_meme_text_rendering.ipynb` pipeline
- **Better emotion alignment** with cleaned, focused text
- **Production-ready** output that fits meme constraints

In [ ]:
def clean_text_for_memes(text, max_words=8):
    """
    Clean and prepare text for meme-style captions.

    Args:
        text (str): Original text
        max_words (int): Maximum words per caption

    Returns:
        str: Cleaned meme-appropriate text
    """
    import re

    # Remove URLs, mentions, hashtags
    text = re.sub(r'http\S+|www\S+|@\w+|#\w+', '', text)

    # Remove extra whitespace and newlines
    text = ' '.join(text.split())

    # Remove quotes and special characters, keep basic punctuation
    text = re.sub(r'["""''`]', '', text)
    text = re.sub(r'[^\w\s,.!?-]', '', text)

    # Split into words and limit length
    words = text.split()
    if len(words) > max_words:
        # Take first part that makes sense
        text = ' '.join(words[:max_words])

    # Ensure it ends properly
    if text and not text[-1] in '.!?':
        if len(words) > max_words:
            text += '...'

    return text.strip().upper()

def split_into_meme_format(text):
    """
    Split longer text into TOP and BOTTOM meme format.

    Args:
        text (str): Input text

    Returns:
        tuple: (top_text, bottom_text) or (text, "") if short
    """
    words = text.split()

    if len(words) <= 4:
        return text, ""

    # Try to split at natural break points
    split_points = []
    for i, word in enumerate(words):
        if word.lower() in ['and', 'but', 'so', 'when', 'then', 'because']:
            split_points.append(i)

    # Use middle split if no natural break found
    if not split_points:
        split_point = len(words) // 2
    else:
        # Use split point closest to middle
        split_point = min(split_points, key=lambda x: abs(x - len(words) // 2))

    top_text = ' '.join(words[:split_point])
    bottom_text = ' '.join(words[split_point:])

    return top_text, bottom_text

def create_meme_training_dataset(df, format_type="meme_format"):
    """
    Convert dataframe to HuggingFace Dataset with meme formatting.

    Args:
        df (pd.DataFrame): DataFrame with mood and caption columns
        format_type (str): "meme_format", "structured", or "simple"

    Returns:
        Dataset: HuggingFace dataset ready for training meme-style captions
    """
    print("Creating meme-formatted training dataset...")

    if format_type == "meme_format":
        # Clean and prepare captions for meme formatting
        print("Applying meme formatting to captions...")

        # Clean the captions
        df["clean_caption"] = df["caption"].apply(clean_text_for_memes)

        # Filter out very short or empty captions
        df = df[df["clean_caption"].str.len() > 3].copy()
        print(f"After meme formatting: {len(df)} cleaned captions remain")

        # Show some examples of before/after meme formatting
        print("\nMeme formatting examples:")
        for i in range(min(3, len(df))):
            original = df.iloc[i]["caption"]
            cleaned = df.iloc[i]["clean_caption"]
            print(f"  Original caption: '{original[:50]}...'")
            print(f"  Cleaned caption:  '{cleaned}'")
            print()

        # Create meme-style caption training data
        training_texts = []
        for _, row in df.iterrows():
            mood = row['mood']
            caption = row['clean_caption']

            # Split into top/bottom if long enough
            top_text, bottom_text = split_into_meme_format(caption)

            if bottom_text:
                # Two-part meme format
                meme_text = f"TOP: {top_text}\nBOTTOM: {bottom_text}"
            else:
                # Single-part meme format (will be TOP only)
                meme_text = f"TOP: {top_text}\nBOTTOM: "

            # Create training prompt
            training_prompt = f"Generate a {mood} meme:\n{meme_text}<|endoftext|>"
            training_texts.append(training_prompt)

        # Create DataFrame for dataset conversion
        training_df = pd.DataFrame({"text": training_texts})

        # Show example of final meme-style caption training format
        print("Meme-style caption training format example:")
        print(training_texts[0])
        print()

    else:
        # Fall back to original function
        return create_training_dataset(df, format_type)

    # Convert to HuggingFace dataset
    dataset = Dataset.from_pandas(training_df)
    print(f"Created meme-formatted dataset with {len(dataset)} training examples")

    return dataset

In [ ]:
# Test the meme formatting on a small sample
print("TESTING MEME FORMATTING")
print("=" * 50)

# Test with a small sample first
test_sample = balanced_df.sample(n=5, random_state=42)

print("BEFORE MEME FORMATTING:")
print("-" * 30)
for i, row in test_sample.iterrows():
    print(f"{row['mood']}: '{row['caption']}'")

print("\nAFTER MEME FORMATTING:")
print("-" * 30)

# Apply meme formatting to see the difference
for i, row in test_sample.iterrows():
    original = row['caption']
    cleaned = clean_text_for_memes(original, max_words=8)
    top_text, bottom_text = split_into_meme_format(cleaned)

    print(f"{row['mood']}:")
    print(f"  Original: '{original}'")
    print(f"  Cleaned:  '{cleaned}'")
    if bottom_text:
        print(f"  Format:   TOP: '{top_text}' | BOTTOM: '{bottom_text}'")
    else:
        print(f"  Format:   TOP: '{top_text}' | BOTTOM: (empty)")
    print()

print("Meme formatting ready, we can use create_meme_training_dataset() for training.")

TESTING MEME FORMATTING
BEFORE MEME FORMATTING:
------------------------------
realization: 'WHEN YOU REALIZE THE MEME YOU MADE SUCKS'
relief: 'ITS A SOFT DRINK'
approval: 'ALRIGHT PEOPLE WE NEED A IDEA FOR A NEW VIDEO GAME! GIRLS WITH BIG BOOBS! MORE GUNS A GOOD STORY LINE'
confusion: 'WHEN THEY THINK I'VE BEEN TELLING AND LAUGH AT MY FAT BISH THAT'S ALL MY BUSINESS'
disappointment: 'BILL WATCHED THE ENTIRE FIRST SEASON OF STRANGER THINGS IN ONE NIGHT OF COURSE, HIS GIRLFRIEND LEFT HIM BECAUSE OF IT NOW HE'S WAITING ANOTHER YEAR FOR SEASON 2 BE LIKE BILL'

AFTER MEME FORMATTING:
------------------------------
realization:
  Original: 'WHEN YOU REALIZE THE MEME YOU MADE SUCKS'
  Cleaned:  'WHEN YOU REALIZE THE MEME YOU MADE SUCKS'
  Format:   TOP: '' | BOTTOM: 'WHEN YOU REALIZE THE MEME YOU MADE SUCKS'

relief:
  Original: 'ITS A SOFT DRINK'
  Cleaned:  'ITS A SOFT DRINK'
  Format:   TOP: 'ITS A SOFT DRINK' | BOTTOM: (empty)

approval:
  Original: 'ALRIGHT PEOPLE WE NEED A IDEA FOR A N

In [ ]:
# Create meme-formatted version of the training dataset
print("Using the same balanced_df, but with meme formatting...")
meme_dataset = create_meme_training_dataset(balanced_df, format_type="meme_format")
meme_tokenized_dataset, meme_tokenizer = tokenize_dataset(meme_dataset)

print("\nDATASET COMPARISON:")
print("-" * 40)
print(f"Original structured dataset: {len(tokenized_dataset)} examples")
print(f"Meme-formatted dataset:     {len(meme_tokenized_dataset)} examples")

print("\nTRAINING FORMAT COMPARISON:")
print("-" * 40)
print("Original Format Example:")
print(dataset[0]['text'][:100] + "..." if len(dataset[0]['text']) > 100 else dataset[0]['text'])

print("\nMeme-Style Caption Format Example:")
print(meme_dataset[0]['text'][:100] + "..." if len(meme_dataset[0]['text']) > 100 else meme_dataset[0]['text'])

# For the training section, we now use meme-formatted dataset
training_dataset = meme_tokenized_dataset
training_tokenizer = meme_tokenizer

Using the same balanced_df, but with meme formatting...
Creating meme-formatted training dataset...
Applying meme formatting to captions...
After meme formatting: 37622 cleaned captions remain

Meme formatting examples:
  Original caption: 'LOOK AT MY FLAPPY BIRD HIGH... SHUT THE F**K UP!...'
  Cleaned caption:  'LOOK AT MY FLAPPY BIRD HIGH... SHUT THE...'

  Original caption: 'IS YOUR NAME WI FI CAUSE I FEEL A CONNECTION...'
  Cleaned caption:  'IS YOUR NAME WI FI CAUSE I FEEL...'

  Original caption: 'PEWDIEPIE LIKE BUTTON...'
  Cleaned caption:  'PEWDIEPIE LIKE BUTTON'

Meme-style caption training format example:
Generate a disapproval meme:
TOP: LOOK AT MY FLAPPY
BOTTOM: BIRD HIGH... SHUT THE...<|endoftext|>

Created meme-formatted dataset with 37622 training examples
Tokenizing dataset...


Map:   0%|          | 0/37622 [00:00<?, ? examples/s]

Tokenized 37622 examples

DATASET COMPARISON:
----------------------------------------
Original structured dataset: 37693 examples
Meme-formatted dataset:     37622 examples

TRAINING FORMAT COMPARISON:
----------------------------------------
Original Format Example:
Generate a disapproval caption: LOOK AT MY FLAPPY BIRD HIGH... SHUT THE F**K UP!<|endoftext|>

Meme-Style Caption Format Example:
Generate a disapproval meme:
TOP: LOOK AT MY FLAPPY
BOTTOM: BIRD HIGH... SHUT THE...<|endoftext|>


<div style="background-color:#e6f2ff; border-left:8px solid #0059b3; padding:20px; margin:20px 0;">
  <h2 style="color:#003366;"><strong>Model Fine-Tuning</strong></h2>
  <p style="color:#333333;">Train GPT-2 on our balanced, meme-formatted dataset to generate emotionally aligned meme-style captions. This section implements the fine-tuning pipeline with checkpoint resume support and optimized training parameters for mood-conditioned text generation.</p>
</div>

In [ ]:
def fine_tune_gpt2(tokenized_dataset, tokenizer, output_dir="../models/gpt2-mood-caption-v2-both",
                   epochs=5, batch_size=4, learning_rate=2e-5, resume_from_checkpoint=None):
    """
    Fine-tune GPT-2 model for mood-conditioned caption generation.

    Args:
        tokenized_dataset (Dataset): Tokenized training dataset
        tokenizer: GPT-2 tokenizer
        output_dir (str): Directory to save the fine-tuned model
        epochs (int): Number of training epochs
        batch_size (int): Training batch size
        learning_rate (float): Learning rate for training
        resume_from_checkpoint (str, optional): Path to checkpoint to resume training from

    Returns:
        Trainer: Trained model trainer object
    """
    print("Starting GPT-2 Fine-Tuning...")

    # Split dataset into train and evaluation sets
    split_dataset = tokenized_dataset.train_test_split(test_size=0.1)
    train_dataset = split_dataset["train"]
    eval_dataset = split_dataset["test"]

    # Load base GPT-2 model
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    print("Loaded base GPT-2 model")

    # Data collator for language modeling
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    # Training arguments - optimized for better fine-tuning
    training_args = TrainingArguments(
        output_dir=output_dir,
        report_to="none",  # Disable wandb logging
        per_device_train_batch_size=batch_size,
        num_train_epochs=epochs,
        learning_rate=learning_rate,
        warmup_steps=200,  # Learning rate warmup for stability
        lr_scheduler_type="cosine",  # Cosine learning rate decay
        save_steps=500,
        save_total_limit=2,
        logging_steps=50,  # More frequent logging
        # evaluation_strategy="steps" if len(tokenized_dataset) > 10000 else "no",
        # eval_steps=1000 if len(tokenized_dataset) > 10000 else None,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
        dataloader_drop_last=True,
        gradient_accumulation_steps=2,  # Effective batch size = 4 * 2 = 8
        adam_epsilon=1e-8,  # More stable optimizer
        max_grad_norm=1.0,  # Gradient clipping
        # load_best_model_at_end=True,
        metric_for_best_model="loss",
        greater_is_better=False,
    )

    # Initialize trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset, # tokenized_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator
    )

    print(f"Training Configuration:")
    print(f"   Dataset size: {len(tokenized_dataset)} examples")
    print(f"   Epochs: {epochs}")
    print(f"   Batch size: {batch_size}")
    print(f"   Learning rate: {learning_rate}")
    print(f"   Output directory: {output_dir}")
    print(f"   Using GPU: {torch.cuda.is_available()}")
    print(f"   Resume from checkpoint: {resume_from_checkpoint if resume_from_checkpoint else 'Starting fresh'}")

    # Start training
    print("\nStarting training...")
    trainer.train(resume_from_checkpoint=resume_from_checkpoint)

    # Ensure models directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Save the fine-tuned model
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    print(f"\nFine-tuning complete! Model saved to {output_dir}")
    return trainer

In [ ]:
def get_checkpoint_or_none(output_dir, subdirectory="checkpoints"):
    """
    Get checkpoint or None with consistent logging.

    This function eliminates repeated checkpoint resume logic by providing
    a single, reusable function for checkpoint detection and resume logic.

    Args:
        output_dir (str): Main output directory (e.g., "../models/gpt2-mood-caption-v2")
        subdirectory (str): Checkpoint subdirectory name (default: "checkpoints")

    Returns:
        str or None: Path to latest checkpoint, or None if no checkpoints found
    """
    if not os.path.exists(output_dir):
        print("No output directory found, cannot search for checkpoints.")
        return None

    # List all entries in the output directory that start with 'checkpoint-' and are directories
    checkpoint_files = [f for f in os.listdir(output_dir)
                        if f.startswith('checkpoint-') and os.path.isdir(os.path.join(output_dir, f))]

    if not checkpoint_files:
        print("No existing checkpoints found — starting from scratch.")
        return None

    # Sort by checkpoint number to find the latest one
    checkpoint_files.sort(key=lambda x: int(x.split('-')[1]))
    latest_checkpoint = os.path.join(output_dir, checkpoint_files[-1])

    print(f"Resuming training from checkpoint: {latest_checkpoint}")
    return latest_checkpoint

In [ ]:
# TRAINING EXECUTION: Using Meme-Formatted Dataset

# Configuration
output_dir = "../models/gpt2-mood-caption-v2-imgflip-both"

# 🔧 DRY Fix: Use helper function instead of repeated checkpoint logic
resume_checkpoint = get_checkpoint_or_none(output_dir)

print("\n" + "=" * 70)

# Start training with meme-formatted dataset
try:
    trainer = fine_tune_gpt2(
        tokenized_dataset=training_dataset,  # Using meme-formatted dataset
        tokenizer=training_tokenizer,        # Using meme-formatted tokenizer
        output_dir=output_dir,
        epochs=5,  # More epochs for better learning
        batch_size=4,
        learning_rate=2e-5,  # Lower learning rate for stability
        resume_from_checkpoint=resume_checkpoint  # Now using DRY helper function
    )

    print("\n" + "=" * 60)
    print("TRAINING COMPLETED SUCCESSFULLY!")
    print("Model trained on meme-formatted dataset!")
    print("Ready for meme-style caption generation.")
    print("=" * 60)

except KeyboardInterrupt:
    print("\n" + "!" * 60)
    print("WARNING: Training was interrupted.")
    print("Training was interrupted. You can resume later.")
    print("!" * 60)

except Exception as e:
    print("\n" + "!" * 60)
    print("ERROR: Training failed.")
    print(f"Training failed with error: {e}")
    print("Check the checkpoint directory for partial progress.")
    print("!" * 60)
    raise e

No output directory found, cannot search for checkpoints.

Starting GPT-2 Fine-Tuning...
Loaded base GPT-2 model
Training Configuration:
   Dataset size: 37622 examples
   Epochs: 5
   Batch size: 4
   Learning rate: 2e-05
   Output directory: ../models/gpt2-mood-caption-v2-imgflip-both
   Using GPU: True
   Resume from checkpoint: Starting fresh

Starting training...


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,4.795800
100,3.325000
150,2.514100
200,2.289500
250,2.175700
300,2.158400
350,2.114800
400,2.074200
450,2.079300
500,2.036500



Fine-tuning complete! Model saved to ../models/gpt2-mood-caption-v2-imgflip-both

TRAINING COMPLETED SUCCESSFULLY!
Model trained on meme-formatted dataset!
Ready for meme-style caption generation.


## Summary & Next Steps

### What We Accomplished

1. **Defined Clear Objectives**: Established controlled text generation as our NLP task with emotional alignment goals
2. **Literature Review**: Justified GPT-2 selection based on research in controlled generation and meme captioning
3. **Baseline Establishment**: Tested zero-shot GPT-2 performance for comparison with fine-tuned model
4. **Modular Implementation**: Created reusable functions for data preparation, tokenization, and model training
5. **Model Training**: Successfully fine-tuned GPT-2 on GoEmotions dataset for mood-conditioned caption generation

### Model Outputs

- **Fine-tuned Model**: `../models/gpt2-mood-caption-v2-imgflip/` (saved to main project models folder)
- **Training Data**: `../data/mood_captions_imgflip.csv` (saved to main project data folder)
- **Baseline Results**: Stored for comparison in next notebook

### Next Steps

The next notebook (`2b_caption_generation.ipynb`) will focus on:
- Loading and using the fine-tuned model for inference
- Comprehensive evaluation metrics (polarity analysis, semantic similarity)
- Comparison between baseline and fine-tuned performance
- Production-ready caption generation functions
- Integration-ready code for the cartoonization pipeline

### Reusability

All functions are modular and documented to support:
- Easy integration into larger systems
- Reproducible model training and inference
- Adaptation for different datasets or model architectures
- Deployment in production environments
